# 03. Feature Selection

**Group 24 - EC5203 Machine Learning - Fetal Well-being Classification (JNU-CTG)**

**Goal:** find out which of the 103 CTG features are redundant or most useful. Everything is calculated on the **training set only**.

**Steps**
1. Settings
2. **Part A** - Pearson correlation of the training features
3. **Part B** - Mutual information + Random Forest ranking, top-k (planned)

In [1]:
# Settings: imports, paths, column names and the random seed.
# These used to live in config/config.py. Each notebook now has its own copy,
# so the project no longer needs the src/ or config/ folders.
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Find the project root: the first folder (going upwards) that contains data/raw.
# This works on every team member's computer - no personal C:/ or F:/ paths.
PROJECT_ROOT = Path.cwd()
for folder in [Path.cwd()] + list(Path.cwd().parents):
    if (folder / "data" / "raw").is_dir():
        PROJECT_ROOT = folder
        break

# Data files
RAW_DATA_FILE = PROJECT_ROOT / "data" / "raw" / "metadata.csv"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_TRAIN_FILE = PROCESSED_DATA_DIR / "train.csv"
PROCESSED_VAL_FILE = PROCESSED_DATA_DIR / "val.csv"
PROCESSED_TEST_FILE = PROCESSED_DATA_DIR / "test.csv"   # used ONCE, only in the final evaluation

# All saved tables go into ONE flat results folder (no separate figures/metrics folders)
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
METRICS_DIR = RESULTS_DIR
FIGURES_DIR = RESULTS_DIR
FEATURE_IMPORTANCE_DIR = RESULTS_DIR

# Column names
TARGET_COLUMN = "figo_label"        # 0 = normal, 1 = abnormal (positive class)
PATIENT_ID_COLUMN = "patient_id"    # used only to group the split, never a feature
RECORD_ID_COLUMN = "record_id"      # used only for traceability, never a feature

# Experiment settings
RANDOM_STATE = 42
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15
CORRELATION_THRESHOLD = 0.90
TOP_K_VALUES = [10, 20, 30, 50]

print("Project root:", PROJECT_ROOT)
print("Random seed: ", RANDOM_STATE)


Project root: c:\Users\Rathishan\Desktop\Desktop\Git\ML_Module\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features
Random seed:  42


---
## Part A - Pearson correlation of the training features

*Moved from `notebooks/08_logistic_regression_correlation.ipynb` - author: Anusikan*

## Pearson correlation of training features


### 1. Set up
Import the libraries and locate the shared training CSV. Check the printed path before continuing.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

relative_path = Path("data/processed/train.csv")
train_path = next(
    (folder / relative_path for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / relative_path).is_file()),
    None,
)
if train_path is None:
    raise FileNotFoundError("Open the repository folder in VS Code before running this notebook.")
print(f"Training data: {train_path}")

Training data: c:\Users\Rathishan\Desktop\Desktop\Git\ML_Module\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\data\processed\train.csv


### 2. Load and check the training data
Select the 103 feature columns by prefix and use `figo_label` as the target. Verify that the values are numeric and finite and that both labels are present. These checks leave the feature values unchanged.

In [3]:
train = pd.read_csv(train_path)
features = [column for column in train if column.startswith(("F_", "U_", "FU_"))]
if len(features) != 103:
    raise ValueError(f"Expected 103 CTG predictors, found {len(features)}.")

X_train = train[features]
y_train = train["figo_label"]
if y_train.isna().any() or set(y_train) != {0, 1}:
    raise ValueError("Expected both FIGO labels, 0 (normal) and 1 (abnormal).")
if not all(pd.api.types.is_numeric_dtype(X_train[column]) for column in features):
    raise ValueError("All CTG predictors must be numeric.")
if not np.isfinite(X_train.to_numpy(dtype=float)).all():
    raise ValueError("The training features contain missing or infinite values.")

print(f"Training shape: {X_train.shape}")
print(y_train.value_counts().sort_index().to_string())

Training shape: (14540, 103)
figo_label
0    6719
1    7821


### 3. Calculate Pearson correlation


In [4]:
corr = X_train.corr(method="pearson")
print(f"Correlation matrix shape: {corr.shape}")
display(corr.iloc[:20, :20].round(3))
print(f"F_Mean vs F_Med: {corr.loc['F_Mean', 'F_Med']:.4f}")

Correlation matrix shape: (103, 103)


,F_BL,F_A_N,F_D_N,F_ED_N,F_VD_N,F_LD_N,F_PD_N,F_LTV,F_STV,F_II,F_DBMM,F_Mean,F_Med,F_Max,F_Min,F_SD,F_MAD,F_MedAD,F_S,F_K
F_BL,1.000,0.035,0.136,0.006,0.151,0.033,0.008,0.973,-0.022,-0.040,0.013,0.974,0.982,0.152,0.085,0.016,0.018,0.077,-0.364,0.076
F_A_N,0.035,1.000,0.207,0.043,0.226,0.030,-0.023,0.147,0.693,0.184,0.684,0.152,0.111,0.286,-0.192,0.446,0.504,0.538,-0.016,-0.124
F_D_N,0.136,0.207,1.000,0.452,0.916,0.458,0.081,0.120,0.448,-0.142,0.494,0.133,0.191,0.255,-0.329,0.514,0.469,0.353,-0.271,-0.022
F_ED_N,0.006,0.043,0.452,1.000,0.131,0.318,-0.004,0.002,0.142,-0.019,0.149,0.004,0.026,0.065,-0.108,0.132,0.119,0.085,-0.089,-0.018
F_VD_N,0.151,0.226,0.916,0.131,1.000,0.166,0.041,0.140,0.448,-0.140,0.496,0.153,0.204,0.261,-0.321,0.512,0.466,0.358,-0.260,-0.011
F_LD_N,0.033,0.030,0.458,0.318,0.166,1.000,0.018,0.019,0.142,-0.052,0.155,0.022,0.055,0.073,-0.112,0.162,0.146,0.099,-0.112,-0.029
F_PD_N,0.008,-0.023,0.081,-0.004,0.041,0.018,1.000,-0.059,0.015,-0.160,0.032,-0.047,-0.004,0.025,-0.105,0.257,0.270,0.141,-0.069,-0.022
F_LTV,0.973,0.147,0.120,0.002,0.140,0.019,-0.059,1.000,0.049,-0.037,0.089,0.999,0.981,0.218,0.112,0.054,0.080,0.167,-0.297,0.036
F_STV,-0.022,0.693,0.448,0.142,0.448,0.142,0.015,0.049,1.000,0.277,0.937,0.061,0.056,0.433,-0.435,0.703,0.654,0.544,-0.139,0.003
F_II,-0.040,0.184,-0.142,-0.019,-0.140,-0.052,-0.160,-0.037,0.277,1.000,0.094,-0.048,-0.045,-0.177,0.119,-0.407,-0.395,-0.260,0.061,-0.026


F_Mean vs F_Med: 0.9831


### 4. Inspect highly correlated pairs
Read only the upper triangle to count each pair once. Use the team threshold of absolute Pearson correlation 0.90. The table preserves the sign and sorts pairs from strongest to weakest. A single feature may appear in several pairs.

In [5]:
threshold = 0.90  # CORRELATION_THRESHOLD in the Settings cell
row, column = np.triu_indices(len(corr), k=1)
pairs = pd.DataFrame({
    "Feature A": corr.index[row],
    "Feature B": corr.columns[column],
    "Pearson r": corr.to_numpy()[row, column],
})
high_corr = pairs.loc[pairs["Pearson r"].abs() >= threshold].sort_values(
    "Pearson r", key=lambda values: values.abs(), ascending=False, kind="stable"
)

print(f"Pairs with |r| >= {threshold:.2f}: {len(high_corr)}")
display(high_corr.head(15).round(4))

Pairs with |r| >= 0.90: 127


,Feature A,Feature B,Pearson r
4264,U_F,U_N,1.0000
3176,F_SE_CWTTF,F_K_CWTTF,-1.0000
2839,F_LPF_CWT10,F_EF_CWTTF,1.0000
2907,F_LPF_CWT11,F_EF_CWTTF,1.0000
2838,F_LPF_CWT10,F_LPF_CWT11,1.0000
3240,F_M_CWTTF,F_C_CWTTF,-1.0000
3111,F_RE_CWTTF,F_S_CWTTF,-1.0000
2625,F_Con_STFT,F_H_STFT,1.0000
2552,F_Cor_STFT,F_Con_STFT,1.0000
2553,F_Cor_STFT,F_H_STFT,1.0000


### 5. Compare targeted filtering with the unscaled baseline

Remove `F_Mean` and `F_BL` while retaining `F_Med`. These summarize FHR level, and their training correlations with the median are checked below. Remove `U_F` while retaining `U_N` because the training values follow `U_F ? U_N / 30`.

Fit both models on the same training rows with identical settings. Compare abnormal-class F1 on training and validation, missed abnormal validation recordings, and changes in individual validation predictions.

In [6]:
import warnings

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score

removed = ["F_Mean", "F_BL", "U_F"]
retained = {"F_Mean": "F_Med", "F_BL": "F_Med", "U_F": "U_N"}
for feature, counterpart in retained.items():
    print(f"{feature} vs {counterpart}: training Pearson r = {corr.at[feature, counterpart]:.6f}")
print(f"Maximum training difference between U_F and U_N / 30: "
      f"{(train['U_F'] - train['U_N'] / 30).abs().max():.3g}")

filtered = [feature for feature in features if feature not in removed]
if len(filtered) != 100:
    raise ValueError("Expected 100 features after removing the three selected columns.")

val = pd.read_csv(train_path.with_name("val.csv"))
if set(train["patient_id"]) & set(val["patient_id"]):
    raise ValueError("Training and validation pregnancies overlap.")
y_val = val["figo_label"]
settings = dict(solver="lbfgs", C=1.0, tol=1e-4, random_state=42, max_iter=1000)
results = {}
validation_predictions = {}

for name, columns in (("Unscaled baseline", features), ("Targeted filter", filtered)):
    model = LogisticRegression(**settings)
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        model.fit(X_train[columns], y_train)

    train_f1 = f1_score(y_train, model.predict(X_train[columns]), pos_label=1)
    predictions = model.predict(val[columns])
    validation_predictions[name] = predictions
    val_f1 = f1_score(y_val, predictions, pos_label=1)
    matrix = confusion_matrix(y_val, predictions, labels=[0, 1])
    results[name] = {
        "Features": len(columns),
        "Training abnormal F1": train_f1,
        "Validation abnormal F1": val_f1,
        "Train minus validation F1": train_f1 - val_f1,
        "Validation false negatives": int(matrix[1, 0]),
    }

comparison = pd.DataFrame.from_dict(results, orient="index")
changed = np.count_nonzero(
    validation_predictions["Unscaled baseline"] != validation_predictions["Targeted filter"]
)
display(comparison.round(4))
print(f"Changed validation labels: {changed} of {len(val)}")

F_Mean vs F_Med: training Pearson r = 0.983107
F_BL vs F_Med: training Pearson r = 0.981562
U_F vs U_N: training Pearson r = 1.000000
Maximum training difference between U_F and U_N / 30: 3.33e-10


,Features,Training abnormal F1,Validation abnormal F1,Train minus validation F1,Validation false negatives
Unscaled baseline,103,0.7756,0.7711,0.0045,426
Targeted filter,100,0.7756,0.7711,0.0045,426


Changed validation labels: 0 of 3115


---
## Part B - Mutual information + Random Forest ranking (planned, not implemented yet)

Method from the approved proposal, fitted on the **training set only**:
1. Absolute Pearson correlation filter, threshold **0.90**, columns in alphabetical order.
2. Rank the remaining features by **mutual information** and by **Random Forest importance**.
3. For **k = 10, 20, 30, 50**: keep the features that are in **both** top-k lists.
4. Train every model on each subset and on all 103 features; compare on the **validation set** (F1 abnormal, then ROC-AUC).